# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 📊 Full Feature Dictionary

#### Core Metadata
*   **report_date**: The date of activity (YYYY-MM-DD).
*   **client_hash_id**: Unique identifier for the website owner (anonymized).
*   **content_hash_id**: Unique identifier for the specific page URL (anonymized).
*   **client_has_gsc / client_has_ga4**: Boolean flags indicating if the source data is connected.
*   **gsc_data_available / ga4_data_available**: Boolean flags for whether data exists for this specific row.

#### Google Search Console (GSC) — Visibility
*   **gsc_impressions**: Number of times the page appeared in search results.
*   **gsc_clicks**: Number of clicks from Google search to the page.
*   **gsc_sum_position**: The sum of rankings (used to calculate average position).
*   **gsc_avg_position**: Average ranking (1 is top, 100 is bottom of page 10).

#### Google Analytics 4 (GA4) — Engagement
*   **ga4_pageviews**: Total times the page was loaded.
*   **ga4_sessions**: Number of distinct browsing sessions.
*   **ga4_users**: Number of unique visitors.
*   **ga4_engaged_sessions**: Sessions that lasted >10 seconds or had a conversion/2+ pageviews.
*   **ga4_total_engagement_sec**: Total active time users spent on the page.
*   **scroll_events**: Count of 'scroll' triggers (usually 90% depth).

#### Traffic Sources & AI Search
*   **sessions_organic / direct / referral / social / paid**: Breakdowns of where the user came from.
*   **sessions_ai**: Total sessions coming from known AI/LLM interfaces.
*   **ai_chatgpt / ai_perplexity / ai_gemini / ai_copilot / ai_claude / ai_meta / ai_other**: Specific breakdown of which AI engine sent the traffic.

In [1]:
# Necessary libraries imported
import pandas as pd

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane:** <br>Freestyle=> First i will do clustering-what kind of items exists and then do classification to classify will the pages of specific kind needs refresh or not.<hr>
Why? Reason: I found this interesting to do.

In [21]:
# Loading the local CSV dataset provided for this task
path = 'content_refresh_anonymized.csv'
data = pd.read_csv(path)

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Justification for the Freestyle Lane (Clustering + Classification)
# 1. Clustering Justification: Check variance in engagement across types
type_engagement = data.groupby('content_type')['sessions_90d'].mean().to_dict()

# 2. Classification Justification: Identify the volume of 'Stale' content
stale_content_pct = (len(data[data['days_since_last_update'] > 180]) / len(data)) * 100

print(f"Average Sessions by Type: {type_engagement}")
print(f"Percentage of content considered 'Stale' (>180 days): {stale_content_pct:.2f}%")
print("Conclusion: Diversity in engagement by type justifies Clustering; high stale % justifies Classification.")

Average Sessions by Type: {'comparison article': 3.624103299856528, 'feedly article': 8.934637404580153, 'keyword article': 40.090638438637114}
Percentage of content considered 'Stale' (>180 days): 0.58%
Conclusion: Diversity in engagement by type justifies Clustering; high stale % justifies Classification.


In [24]:
# Detailed Breakdown: Stale content distribution by content_type
# This helps determine if specific clusters (types) need priority attention.

stale_threshold = 180
stale_by_type = data[data['days_since_last_update'] > stale_threshold]['content_type'].value_counts()
total_by_type = data['content_type'].value_counts()

# Calculate percentage of staleness per category
stale_pct_by_type = (stale_by_type / total_by_type * 100).fillna(0)

print(f"--- Stale Content Breakdown (> {stale_threshold} days) ---")
for c_type in total_by_type.index:
    count = stale_by_type.get(c_type, 0)
    pct = stale_pct_by_type.get(c_type, 0)
    print(f"{c_type}: {count} stale pages ({pct:.2f}% of category)")

--- Stale Content Breakdown (> 180 days) ---
keyword article: 174 stale pages (0.64% of category)
feedly article: 0 stale pages (0.00% of category)
comparison article: 0 stale pages (0.00% of category)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:**<br>
It helps decide where to put our resources specifically, instead of fixing everything it fixes the specific flagged page or specific flagged pages inside specific cluster.<hr>
**Improvement:**<br>
This saves time of manual looking into pages and their content types and automate the process of clustering relevant content and flag if needed refresh. This saves a lot of time and money and provide good results as compare to human-rule involvement. <br>
**Who acts?**<br>
A company/individual who has limited budget and limited time and who wants frequent traffic on their content to run their businesses or want frequent engagement on their sites.
<hr>
**Wrong recommendation costs**<br>
False Positive (Type I): We recommend a refresh for a page that was actually doing fine. Cost: Wasted salary/time on unnecessary work.
False Negative (Type II): We miss a page that is dying. Cost: Permanent loss of traffic and revenue as Google stops indexing the outdated content in favor of competitors.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Type I Risk (Waste): What is the average traffic 'at stake' per page?
avg_sessions_per_page = data['sessions_90d'].mean()

# 2. Type II Risk (Loss): Total traffic currently sitting in 'stale' content
# This is traffic that is highly likely to decay further without action.
stale_traffic_total = data[data['days_since_last_update'] > 180]['sessions_90d'].sum()
stale_traffic_pct = (stale_traffic_total / data['sessions_90d'].sum()) * 100

print(f"--- Business Risk ---")
print(f"Type I (Waste) Risk: Each unnecessary refresh targets a page averaging {avg_sessions_per_page:.1f} sessions/90d.")
print(f"Type II (Loss) Risk: {stale_traffic_total:,.0f} sessions ({stale_traffic_pct:.2f}% of total) are currently 'stale'.")

--- Business Risk ---
Type I (Waste) Risk: Each unnecessary refresh targets a page averaging 37.1 sessions/90d.
Type II (Loss) Risk: 964 sessions (0.09% of total) are currently 'stale'.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
path = 'content_refresh_anonymized.csv'
data = pd.read_csv(path)

In [30]:
print(data.columns)

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')


In [31]:
# Displaying real numbers for Section 3
print(f"Dataset loaded with {data.shape[0]} rows and {data.shape[1]} columns.")
display(data.head(10))

Dataset loaded with 30000 rows and 44 columns.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [32]:
# 1. Diversity: Unique Content Types & Intents
unique_types = data['content_type'].nunique()
unique_intents = data['main_intent'].nunique()

# 2. Potential Decay: Stale pages (e.g., > 180 days) with significant impressions but low CTR
stale_mask = (data['days_since_last_update'] > 180) & (data['impressions_90d'] > 100)
stale_count = len(data[stale_mask])

# 3. Scale of the Problem: Total sessions tied to 'old' content
total_sessions_at_risk = data[data['days_since_last_update'] > 180]['sessions_90d'].sum()

print(f"--- Justification Numbers for Section 3 ---")
print(f"1. Content Diversity: {unique_types} types and {unique_intents} intents (Requires Clustering)")
print(f"2. Refresh Candidates: {stale_count} high-visibility pages are >6 months old (Requires Classification)")
print(f"3. Resource Scale: {total_sessions_at_risk:,.0f} sessions are coming from 'stale' content (Business Value)")

--- Justification Numbers for Section 3 ---
1. Content Diversity: 3 types and 4 intents (Requires Clustering)
2. Refresh Candidates: 35 high-visibility pages are >6 months old (Requires Classification)
3. Resource Scale: 964 sessions are coming from 'stale' content (Business Value)


In [18]:
print(data['content_type'].value_counts())

content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


In [19]:
print(data['main_intent'].value_counts())

main_intent
informational    17235
transactional     5733
commercial        4612
navigational        46
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

My work is basically a decision-support. It will classify the decision and support in solution to the content optimization problems. Despite the model working, its still preferable to verify the decision by human involvement who are expert in this field. <hr>

**It will never:**<br>
->Predict Google's core algorithm changes: I can only see how the site reacted and how should it react, not what Google will do next.<br>
->Claim Causal Proof: I can show that traffic dropped when content wasn't refreshed, but I can't prove it was the only reason.<br>
->Guarantee Rankings: I can suggest improvements, but I cannot predict or control specific ranking positions.<br>
-> Auto refresh the page contents, it will only provide a decision whether to refresh or not.<br>
-> Fix SEO issues and its errors<br>
-> Identify which user visits your page<br>
-> Claim that decision made by model is 100% certain because model trained on data which is of a limited time period not whole year. <br>

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 1. Statistical Significance Risk: How many pages have very low data volume?
low_data_threshold = 10
low_data_pages = data[data['impressions_90d'] < low_data_threshold]
low_data_pct = (len(low_data_pages) / len(data)) * 100

# 2. Seasonality/Technical Gaps: Identify missing signals
# Checking for technical SEO columns or longer timeframes
columns_present = data.columns.tolist()
technical_signals = ['site_speed', 'crawl_errors', 'server_response_time']
has_technical = any(col in columns_present for col in technical_signals)

print(f"--- Limitation Justification ---")
print(f"1. Statistical Risk: {low_data_pct:.2f}% of pages have < {low_data_threshold} impressions in 90 days. We cannot claim high confidence for these specific rows.")
print(f"2. Technical Blindspot: Found 0 technical SEO metrics ({technical_signals}). We cannot claim a refresh fixes infrastructure issues.")
print(f"3. Timeframe Constraint: Data spans 90 days. Seasonal cycles (annual) cannot be statistically distinguished from content decay.")

--- Limitation Justification ---
1. Statistical Risk: 12.49% of pages have < 10 impressions in 90 days. We cannot claim high confidence for these specific rows.
2. Technical Blindspot: Found 0 technical SEO metrics (['site_speed', 'crawl_errors', 'server_response_time']). We cannot claim a refresh fixes infrastructure issues.
3. Timeframe Constraint: Data spans 90 days. Seasonal cycles (annual) cannot be statistically distinguished from content decay.


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.